# Get company name from text



In [16]:
import re

DATE_PREFIX_RE = re.compile(
    r"""^\s*
        (?:                                   # common date prefixes
            \d{1,2}\s*[A-Za-z]{3,}\w*\s*\d{2,4}   # 13 febv 26 / 17 Feb 2026
          | \d{1,2}[-/]\d{1,2}[-/]\d{2,4}         # 13-02-2026 / 13/02/26
        )
        \s*[-–—:]*\s*                          # optional separators after date
    """,
    re.IGNORECASE | re.VERBOSE
)

def extract_company_name_from_bse_text(text: str) -> str | None:
    # Find first 6-digit BSE scrip code
    m = re.search(r"\b\d{6}\b", text)
    if not m:
        return None
    company_code = m.group(0)

    # Take only the same-line chunk immediately before the code
    line_start = max(text.rfind("\n", 0, m.start()), text.rfind("\r", 0, m.start()))
    line_start = 0 if line_start == -1 else line_start + 1
    candidate = text[line_start:m.start()].strip()

    # Remove a leading date prefix if present (e.g., "13 febv 26- ")
    candidate = DATE_PREFIX_RE.sub("", candidate).strip()

    # Clean trailing separators / currency artifacts
    candidate = re.sub(r"[-\s]+$", "", candidate)
    candidate = re.sub(r"[-\s]*[₹$]+$", "", candidate)
    candidate = re.sub(r"[-\s]+$", "", candidate).strip()

    if candidate:
        return company_code + "_" + candidate
    elif company_code:
        return company_code + "_"
    else:
        return text.strip()  # fallback to raw text if no code found


tests = [
"13 febv 26 \n Desco Infratech Ltd - 544387 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"13 febv 26- Gabion Technologies India Ltd - 544675 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"HFCL Ltd - 500183 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"Highway Infrastructure Ltd - 544477 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"Seamec Ltd - 526807 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order",
"17 Feb 2026\nHazoor Multi Projects Ltd-$ - 532467 - Announcement under Regulation 30 (LODR)-Award_of_Order_Receipt_of_Order ..."
]

for t in tests:
    print(extract_company_name_from_bse_text(t))

544387_Desco Infratech Ltd
544675_Gabion Technologies India Ltd
500183_HFCL Ltd
544477_Highway Infrastructure Ltd
526807_Seamec Ltd
532467_Hazoor Multi Projects Ltd


# get company data

In [59]:
import os
from datetime import datetime, timedelta

def get_pdf_files(before_days: int = 0):
    # Get today's date in yyyy-mm-dd format
    today_date = datetime.now().strftime('%Y-%m-%d')


    # Subtract n day using timedelta
    days_back = datetime.now() - timedelta(before_days)

    # Format the resulting datetime object into a YYYY-MM-DD string
    days_back_formatted = days_back.strftime('%Y-%m-%d')

    # Path to the folder
    folder_path = f'bse_award_orders/{days_back_formatted}'
    
    # List all files in the folder
    files = os.listdir(folder_path)
    
    # Filter to get only the PDF files
    pdf_files = [file for file in files if file.endswith('.pdf')]
    
    return pdf_files

pdf_files = get_pdf_files(before_days=0)
print(pdf_files)

['543806_ITCONS E-Solutions Ltd.pdf']


In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

class ScreenerAutomation:
    def __init__(self):
        # Initialize WebDriver (This automatically uses the ChromeDriver installed)
        self.driver = webdriver.Chrome()

    def fetch_company_data(self, company_code, company_name):
        # Try to open company page
        url = f"https://www.screener.in/company/{company_code}/"
        self.driver.get(url)

        try:
            # Wait for page to load or give a timeout
            WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

            # Check for 404 error by looking for page elements
            if "404 Not Found" in self.driver.title:
                # Page not found, search for the company
                self.driver.get("https://www.screener.in/")
                search_box = self.driver.find_element(By.ID, "id_search_input")
                search_box.send_keys(company_name)
                search_box.send_keys(Keys.RETURN)

                # Wait for the search results to load and click the first result
                WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul#search-results li a")))
                first_result = self.driver.find_element(By.CSS_SELECTOR, "ul#search-results li a")
                first_result.click()

                # Wait for the company page to load
                WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

        except Exception as e:
            print(f"Error occurred while fetching data for {company_name} with code {company_code}: {e}")
            return None

        # Now extract data from the page
        return self.extract_company_data()

    def extract_company_data(self):
        try:
            # Ensure we are on the "Profit & Loss" tab
            profit_loss_tab = self.driver.find_element(By.XPATH, "//a[contains(text(), 'Profit & Loss')]")
            profit_loss_tab.click()  # Click to focus on the "Profit & Loss" tab

            # Wait for the "Profit & Loss" section to load
            WebDriverWait(self.driver, 10).until(EC.presence_of_element_located((By.XPATH, "//table[contains(@class, 'data-table')]")))

            # Locate the "Net Profit" expand button
            expand_button = self.driver.find_element(By.XPATH, "//button[contains(text(), 'Net Profit')]")
            
            # Scroll into view to make sure it is clickable
            self.driver.execute_script("arguments[0].scrollIntoView(true);", expand_button)

            # Wait to ensure it is clickable
            WebDriverWait(self.driver, 10).until(EC.element_to_be_clickable(expand_button))
            
            # Use JavaScript to click if normal click is blocked
            self.driver.execute_script("arguments[0].click();", expand_button)

            # Wait for the page to update and the table to load with the necessary data
            time.sleep(3)  # Sleep for a few seconds to let the data load

            # Get the HTML of the entire page
            html = self.driver.page_source

            # Pass the HTML to BeautifulSoup for parsing
            soup = BeautifulSoup(html, "html.parser")
            # Refined XPath queries
            market_cap = self.driver.find_element(By.XPATH, '//*[contains(text(), "Market Cap")]/following-sibling::span').text
            roce = self.driver.find_element(By.XPATH, '//*[contains(text(), "ROCE")]/following-sibling::span').text
            roe = self.driver.find_element(By.XPATH, '//*[contains(text(), "ROE")]/following-sibling::span').text
            # net_profit_ttm = self.driver.find_element(By.XPATH, '/html/body/main/section[4]/div[3]/table/tbody/tr[1]/td[14]').text
            # Now extract the Net Profit TTM value after expanding the section
            # net_profit_ttm = self.driver.find_element(By.XPATH, '//tr[contains(@class, "strong")]//td[contains(text(), "Net Profit")]/following-sibling::td[last()]').text

            # Return extracted data as a dictionary
            return {
                "Market Cap": market_cap,
                "ROCE": roce,
                "ROE": roe,
                # "Net Profit TTM": net_profit_ttm
            }
        except Exception as e:
            print(f"Error while extracting company data: {e}")
            return None

    def close(self):
        # Close the browser once done
        self.driver.quit()
import os
from datetime import datetime

# Step 1: Get all PDF files from the folder

# Step 2: Process each file and extract company data
def process_pdf_files(pdf_files):
    screener = ScreenerAutomation()
    
    for pdf_file in pdf_files:
        # Extract company code and company name from PDF filename
        company_code, company_name = pdf_file.replace('.pdf', '').split('_')
        
        print(f"Processing {company_name} with code {company_code}")
        company_data = screener.fetch_company_data(company_code, company_name)
        
        if company_data:
            print(f"Company Data for {company_name}: {company_data}")
        else:
            print(f"Failed to retrieve data for {company_name}")
    
    # Close the browser once all data is processed
    screener.close()

# Get PDF files and process them
pdf_files = get_pdf_files(before_days=1)
process_pdf_files(pdf_files[:2])

# LLM grok


In [2]:
! pip install -U langchain langchain-xai

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached ormsgpack-1.12.2-cp311-cp311-win_amd64.whl.metadata (3.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached orjson-3.11.9-cp311-cp311-win_amd64.whl.metadata (43 kB)
  Using cached websockets-15.0.1-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cac


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
! pip install langchain langchain-openai


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ! pip install langchain langchain-openai langchain-community duckduckgo-search ddgszz
! pip install -U pypdf langgraph langchain-openai



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent

load_dotenv()  # OPENROUTER_API_KEY from .env

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",  # free models often fail at tool loops
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)

ddg = DuckDuckGoSearchRun()

@tool
def company_revenue(company_name: str) -> str:
    """Search the internet for a company's latest annual revenue."""
    query = f"{company_name} latest annual revenue FY"
    print("Searching:", query)
    return ddg.invoke(query)


# ReAct agent: think -> call tool -> observe -> answer
agent = create_react_agent(
    model=llm,
    tools=[company_revenue],
    prompt=(
        "You are a helpful research assistant. "
        "Always use the company_revenue tool before answering revenue questions. "
        "Analyse the revenue number of the given copany"
        "Return only the revenue number"

    ),
)

question = "What is NCC Ltd latest annual revenue?"

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

# Final assistant message
print(result["messages"][-1].content)

# Optional: see the full tool loop
for msg in result["messages"]:
    print(type(msg).__name__, "->", getattr(msg, "content", None) or getattr(msg, "tool_calls", None))

C:\Users\omkar.patil\AppData\Local\Temp\ipykernel_7024\780744132.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(
could not parse an IP from hosts file ("\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0

Searching: NCC Ltd latest annual revenue FY


could not parse an IP from hosts file ("\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0\0

Searching: NCC Limited latest annual revenue FY
NCC Ltd's latest annual revenue is ₹20,944.40 Crore.
HumanMessage -> What is NCC Ltd latest annual revenue?
AIMessage -> [{'name': 'company_revenue', 'args': {'company_name': 'NCC Ltd'}, 'id': 'call_WU6z8ulOZmqfdCmp7QoWObOJ', 'type': 'tool_call'}]
ToolMessage -> Current and historical revenue information for Amazon.com, Inc. (AMZN) stock, including a chart and comparison to related stocks. Annual Reports and Proxy Statements. Our annual report provides a transparent update on our progress and priorities. A decorative graphic for the Citi 2025 Annual Report, featuring the number 25 in a large, stylized outline font against a background of wavy lines. annual sales going from $350 million to more than $1 billion. Data provided by C-MOTS Internet Technologies Pvt Ltd. Tata Consultancy Services (TCS) delivered another year of steady growth in FY2025-26, reporting higher revenue, stronger profitability and industry-leading margins. Yet, the amo

In [6]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


@tool
def read_award_pdf(pdf_path: str) -> str:
    """
    Read text from a BSE Award of Order PDF given its full file path.
    Use this before extracting order value or payment details.
    """
    path = Path(pdf_path)
    if not path.exists():
        return f"ERROR: File not found: {pdf_path}"
    if path.suffix.lower() != ".pdf":
        return f"ERROR: Not a PDF file: {pdf_path}"

    reader = PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append(f"--- Page {i} ---\n{text.strip()}")

    full_text = "\n\n".join(pages).strip()
    if not full_text:
        return "ERROR: No extractable text found. PDF may be scanned/image-only."

    # Keep tool output bounded for the LLM
    return full_text[:15000]


agent = create_react_agent(
    model=llm,
    tools=[read_award_pdf],
    prompt=(
        "You are an analyst for BSE 'Award of Order / Receipt of Order' announcements.\n"
        "Always call read_award_pdf with the exact PDF path from the user before answering.\n"
        "Extract ONLY from the PDF text. If something is not mentioned, say 'Not disclosed'.\n\n"
        "Return the answer in this exact format:\n"
        "Company:\n"
        "Order Value:\n"
        "Currency / GST note:\n"
        "Order breakup (if any):\n"
        "Client / Awarding authority:\n"
        "Project / Scope summary:\n"
        "Payment structure:\n"
        "- One-time payment: Yes/No/Not disclosed\n"
        "- Upfront / advance %: \n"
        "- Milestone / progressive billing: \n"
        "- Retention / BG / LD terms: \n"
        "- Payment cycle / credit period: \n"
        "Contract duration / execution period:\n"
        "Other commercial terms:\n"
        "Notes / caveats:\n"
    ),
)

pdf_path = r"C:\Users\omkar.patil\Desktop\Trantor Project\Stocks_daily_script\bse_award_orders\2026-07-31\500294_NCC Ltd.pdf"

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Analyze this Award of Order PDF and extract order value "
                    f"and payment structure details.\nPDF path: {pdf_path}"
                ),
            }
        ]
    }
)

print(result["messages"][-1].content)

C:\Users\omkar.patil\AppData\Local\Temp\ipykernel_7024\3512208685.py:47: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Company: NCC Limited  
Order Value: Rs.1052.71 Crore  
Currency / GST note: Excl. GST  
Order breakup (if any):  
- Buildings Division: Rs.590.38 Crore  
- Water Division: Rs.462.33 Crore  
Client / Awarding authority: Not disclosed  
Project / Scope summary: Not disclosed  
Payment structure:  
- One-time payment: Not disclosed  
- Upfront / advance %: Not disclosed  
- Milestone / progressive billing: Not disclosed  
- Retention / BG / LD terms: Not disclosed  
- Payment cycle / credit period: Not disclosed  
Contract duration / execution period: Not disclosed  
Other commercial terms: Not disclosed  
Notes / caveats: The orders received do not include any internal order and are not related party transactions.


In [4]:
i = [2,3,[6,8],[9,[45,3],8]]

def flaten(i):
    return [x for item in i for x in (flaten(item) if isinstance(item,list) else [item])]

flaten(i)

[2, 3, 6, 8, 9, 45, 3, 8]